# Validazione del modello di penetrazione (`penetration_lite`)

Per ogni SKU registriamo: **ultimate penetration** (K composta del fit piecewise al cutoff di training), **penetrazione osservata** al periodo di riferimento, **PWSD holdout / full** e la **CV di stabilità** di K.

Flusso: una sola `build_penetration` sull'intera finestra → `validate` fa lo split al cutoff e restituisce sia il fit troncato sia le PWSD → il plot per le slide sovrappone il fit al cutoff alla serie osservata completa → `stability_piecewise` (limitata al training) dà la CV.

In [ ]:
from penetration_lite import (
    build_penetration, fit_piecewise
    , plot_penetration, stability_piecewise, validate
)

## Parametri per SKU

L'unica data "hard" necessaria è la fine della finestra di holdout (`end_date_cutoff_plus1y`, passata come `analysis_date`); il cutoff di training entra come **label di calendario**, che la libreria risolve da sola sull'asse a quindicine.

In [ ]:
# Defined upstream in the work environment:
#   tr_trb                 -> Spark transaction log (category incl. the SKU)
#   launch_date            -> SKU launch date
#   end_date_cutoff_plus1y -> end of the holdout window (training cutoff + 1 year)

promo_labels          = ['2024-W10']  # promo waves, as calendar labels
training_cutoff_label = '2024-W16'    # ISO-week label of end_date_cutoff: validate splits train/holdout here
obs_period            = 43            # period whose observed penetration is recorded
N_STAB                = 4             # trailing cutoffs feeding the stability CV
MIN_STAB_PERIODS      = 6             # first cutoff eligible for the stability scan

## Una sola curva sull'intera finestra

Con denominatore dinamico i P(t) dei periodi ≤ cutoff non cambiano allungando la finestra: troncare la serie equivale a costruirla su meno dati. Basta quindi **una** `build_penetration` (una sola passata Spark) su training + holdout; lo split lo fa `validate`.

In [ ]:
full = build_penetration(
    tr_trb
    , unit='iso_fortnight'
    , launch_date=launch_date
    , denominator='dynamic'
    , analysis_date=end_date_cutoff_plus1y   # inclusive upper bound: training + holdout
)

print(f'{len(full.series)} periods, snapshot P = {full.snapshot:.4f}')

## Fit al cutoff + PWSD

`validate` tronca la serie al cutoff, fitta la piecewise promo-aware sui soli dati di training e proietta sull'intero orizzonte: `pwsd_holdout` misura l'errore sulla sola coda non vista, `pwsd_full` su tutta la serie. `val.curve` **è** il fit piecewise di training: la ultimate penetration registrata viene da lì, così K e PWSD descrivono lo stesso modello.

In [ ]:
val = validate(full, cutoff_period=training_cutoff_label, promo_periods=promo_labels)
pw = val.curve                              # piecewise fit on training data only

print('ultimate_penetration :', pw.ultimate_penetration)
print('pwsd_holdout         :', val.pwsd_holdout)
print('pwsd_full            :', val.pwsd_full)
if val.note:
    print('note                 :', val.note)

display(val.to_frame())                     # actual vs forecast, period by period

## Plot per le slide

Serie osservata completa (training + holdout) con sovrapposto il fit stimato al cutoff: a destra della linea tratteggiata arancione è forecast puro contro dati mai visti dal modello.

In [ ]:
ax = plot_penetration(full, piecewise=pw,
                      title='Cumulative penetration - fit at training cutoff')
ax.axvline(val.cutoff_period, ls=':', color='tab:orange', lw=1.2)
ax.annotate('training cutoff', xy=(val.cutoff_period, 0), xytext=(4, 6),
            textcoords='offset points', rotation=90, fontsize=8,
            color='tab:orange', va='bottom')

# Slide with the most up-to-date estimate instead (fit on the whole window):
# pw_full = fit_piecewise(full, promo_labels)
# plot_penetration(full, piecewise=pw_full)

## Stabilità di K → CV

`stability_piecewise` rifitta la composizione a ogni cutoff. I cutoff sono limitati alla finestra di training (niente lookahead) e le righe di fallback — ultimo segmento non fittato, K mancante — vanno escluse prima della CV: non sono stabilità.

In [ ]:
stab_df = stability_piecewise(
    full
    , promo_periods=promo_labels
    , cutoffs=[t for t, _ in full.series
               if MIN_STAB_PERIODS <= t <= val.cutoff_period]   # training window only
)

# keep only cutoffs where every segment is genuinely fitted (no fallback rows)
ok = stab_df[stab_df['n_segments_fitted'] == len(promo_labels) + 1].dropna(subset=['K'])

tail = ok.tail(N_STAB)['K']                 # 'K' is the composed ultimate penetration
if len(tail) < N_STAB:
    print(f'warning: only {len(tail)} fitted cutoffs available for the CV')
stab_index_cv = tail.std() / tail.mean()

print('stability CV :', stab_index_cv)
display(stab_df.tail(8))

## Riepilogo per SKU

Le cinque metriche registrate, più la `note` — essenziale in batch: segnala fit instabili (K sotto l'osservato), PWSD non calcolabili, promo scartate perché oltre il cutoff.

In [ ]:
sku_metrics = {
    'ultimate_penetration': pw.ultimate_penetration        # composed K, fit at the cutoff
    , 'observed_P'        : dict(full.series).get(obs_period)
    , 'observed_period'   : full.label(obs_period)
    , 'pwsd_holdout'      : val.pwsd_holdout
    , 'pwsd_full'         : val.pwsd_full
    , 'penetration_cv'    : stab_index_cv
    , 'note'              : val.note
}
sku_metrics